# PART 04 · 4장 상관분석과 회귀분석

교재 356~376쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 컬럼의 역할

#### 2. 장 전체의 분석 흐름

#### 3. 공통 라이브러리와 데이터 불러오기

다음 코드는 이 장에서 사용할 데이터를 한 번만 불러온다.

In [1]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
path = (
    'data/part04_type3/correlation_regression_ad_orders/'
    'ad_orders.csv'
)
df = pd.read_csv(path)
print(df.shape)
print(df.head())

(80, 4)
         date  ad_budget  orders  is_weekend
0  2026-01-01      139.5     326           0
1  2026-01-02      114.6     323           0
2  2026-01-03       80.5     256           1
3  2026-01-04       41.8     160           1
4  2026-01-05       40.6     145           0


## 01 상관계수

### 1 상관계수의 해석

#### 1. 상관계수의 범위와 해석

#### 2. 상관계수 행렬 확인

세 컬럼의 상관계수 행렬을 확인해본다.

In [2]:
cols = ['ad_budget', 'orders', 'is_weekend']
corr_matrix = df[cols].corr()
print(corr_matrix.round(4))

            ad_budget  orders  is_weekend
ad_budget      1.0000  0.9464     -0.0913
orders         0.9464  1.0000      0.0543
is_weekend    -0.0913  0.0543      1.0000


### 2 상관관계와 인과관계

#### 1. 상관관계와 인과관계

## 02 피어슨 상관분석

### 1 가설과 실행

#### 1. 귀무가설과 대립가설

#### 2. pearsonr( ) 사용

pearsonr( )로 상관계수와 p-value를 확인하고 t 통계량도 직접 계산해본다.

In [3]:
import numpy as np
x = df['ad_budget']
y = df['orders']
pearson_result = stats.pearsonr(x, y)
r = pearson_result.statistic
p_value = pearson_result.pvalue
n = len(x)
df_value = n-2
t_statistic = r*np.sqrt(df_value/(1-r**2))
print('n: ', n)
print('pearson_r: ', round(r, 4))
print('t_statistic: ', round(t_statistic, 4))
print('df: ', df_value)
print('p_value: ', format(p_value, '.12e'))

n:  80
pearson_r:  0.9464
t_statistic:  25.8681
df:  78
p_value:  5.086418398891e-40


### 2 결론 작성

#### 1. 결론 작성

## 03 스피어만 상관분석

### 1 스피어만의 특징

#### 1. 피어슨과 스피어만의 차이

### 2 실행과 동률 순위

#### 1. spearmanr( ) 사용

spearmanr( )를 실행하고 순위 변환이 어떻게 이루어지는지 확인해본다.

In [4]:
x = df['ad_budget']
y = df['orders']
spearman_result = stats.spearmanr(x, y)
rank_df = pd.DataFrame({
    'ad_budget': x,
    'ad_rank': x.rank(method='average'),
    'orders': y,
    'order_rank': y.rank(method='average')
})
print(rank_df.head())
print('spearman_rho: ', round(spearman_result.statistic, 4))
print('p_value: ', format(spearman_result.pvalue, '.12e'))

   ad_budget  ad_rank  orders  order_rank
0      139.5     77.0     326        64.0
1      114.6     63.0     323        62.5
2       80.5     29.0     256        30.5
3       41.8      5.0     160         4.0
4       40.6      4.0     145         2.0
spearman_rho:  0.9492
p_value:  6.631854637906e-41


#### 2. 동률 순위

## 04 단순선형 회귀분석

### 1 회귀 식과 모형 생성

#### 1. 회귀 식의 구조

#### 2. 단순회귀 모형 생성

광고비 하나를 독립변수로 하는 단순회귀 모형을 만들어 본다.

In [5]:
X_simple = sm.add_constant(df[['ad_budget']])
y = df['orders']
model_simple = sm.OLS(y, X_simple).fit()
print('intercept: ', format(model_simple.params['const'], '.4f'))
print('slope: ', format(model_simple.params['ad_budget'], '.4f'))
print('r_squared: ', format(model_simple.rsquared, '.4f'))
print('adj_r_squared: ', format(model_simple.rsquared_adj, '.4f'))
print('slope_t: ', format(model_simple.tvalues['ad_budget'], '.4f'))
print(
    'slope_p_value: ',
    format(model_simple.pvalues['ad_budget'], '.12e')
)
print('f_statistic: ', format(model_simple.fvalue, '.4f'))
print('f_p_value: ', format(model_simple.f_pvalue, '.12e'))

intercept:  90.9920
slope:  1.9890
r_squared:  0.8956
adj_r_squared:  0.8943
slope_t:  25.8681
slope_p_value:  5.086418398891e-40
f_statistic:  669.1582
f_p_value:  5.086418398891e-40


### 2 해석과 예측

#### 1. 회귀 식과 계수 해석

#### 2. 새로운 값 예측

만든 모형으로 광고비가 100일 때의 주문 수를 예측해본다.

In [6]:
new_X = pd.DataFrame({
    'const': [1],
    'ad_budget': [100]
})
pred = model_simple.predict(new_X)
print(round(pred.iloc[0], 4))

289.8932


## 05 다중선형 회귀분석

### 1 다중 회귀 식과 모형 생성

#### 1. 다중회귀 모형 생성 1

#### 2. 다중회귀 모형 생성 2

광고비와 주말 여부 두 개를 독립변수로 하는 다중회귀 모형을 만들어 본다.

In [7]:
X_multi = sm.add_constant(
    df[['ad_budget', 'is_weekend']]
)
y = df['orders']
model_multi = sm.OLS(y, X_multi).fit()
print(model_multi.params.round(4))
print('r_squared: ', round(model_multi.rsquared, 4))
print('adj_r_squared: ', round(model_multi.rsquared_adj, 4))
print('f_statistic: ', round(model_multi.fvalue, 4))
print('f_p_value: ', format(model_multi.f_pvalue, '.12e'))
print('p_values')
print(model_multi.pvalues)

const         83.0955
ad_budget      2.0163
is_weekend    18.9344
dtype: float64
r_squared:  0.9156
adj_r_squared:  0.9134
f_statistic:  417.5148
f_p_value:  4.672488377688e-42
p_values
const         1.337511e-19
ad_budget     4.953174e-43
is_weekend    5.571224e-05
dtype: float64


### 2 해석과 예측

#### 1. 회귀 식과 계수 해석

#### 2. 평일과 주말 예측값

같은 광고비에서 평일과 주말의 예측값을 비교해본다.

In [8]:
new_X = pd.DataFrame({
    'const': [1, 1],
    'ad_budget': [100, 100],
    'is_weekend': [0, 1]
})
pred = model_multi.predict(new_X)
print('weekday: ', round(pred.iloc[0], 4))
print('weekend: ', round(pred.iloc[1], 4))

weekday:  284.7206
weekend:  303.6549


## 06 회귀계수 해석

### 1 변수 유형별 계수

#### 1. 연속형 변수의 계수

#### 2. 0·1 변수의 계수

### 2 계수 확인과 해석

#### 1. 예측값 차이로 계수 확인

입력값을 1씩 바꾼 예측값의 차이가 회귀계수와 같은지 확인해본다.

In [9]:
new_data = pd.DataFrame({
    'const': [1, 1, 1],
    'ad_budget': [100, 101, 100],
    'is_weekend': [0, 0, 1]
})
pred = model_multi.predict(new_data)
print('budget100_weekday: ', round(pred.iloc[0], 4))
print('budget101_weekday: ', round(pred.iloc[1], 4))
print('budget100_weekend: ', round(pred.iloc[2], 4))
print('budget_1_effect: ', round(pred.iloc[1]-pred.iloc[0], 4))
print('weekend_effect: ', round(pred.iloc[2]-pred.iloc[0], 4))

budget100_weekday:  284.7206
budget101_weekday:  286.7368
budget100_weekend:  303.6549
budget_1_effect:  2.0163
weekend_effect:  18.9344


#### 2. 절편 해석

#### 3. 계수와 p-value

## 07 결정계수 해석

### 1 결정계수 비교

#### 1. 단순회귀와 다중회귀 비교

상관계수의 제곱과 단순회귀의 결정계수, 다중회귀의 결정계수를 비교해본다.

In [10]:
corr = df['ad_budget'].corr(df['orders'])
print('pearson_corr: ', format(corr, '.4f'))
print('corr_squared: ', format(corr**2, '.4f'))
print('simple_r_squared: ', format(model_simple.rsquared, '.4f'))
print(
    'simple_adj_r_squared: ',
    format(model_simple.rsquared_adj, '.4f')
)
print('multi_r_squared: ', format(model_multi.rsquared, '.4f'))
print(
    'multi_adj_r_squared: ',
    format(model_multi.rsquared_adj, '.4f')
)
print(
    'r_squared_increase: ',
    format(model_multi.rsquared - model_simple.rsquared, '.4f')
)

pearson_corr:  0.9464
corr_squared:  0.8956
simple_r_squared:  0.8956
simple_adj_r_squared:  0.8943
multi_r_squared:  0.9156
multi_adj_r_squared:  0.9134
r_squared_increase:  0.0200


#### 2. 수정 결정계수

### 2 상관계수 제곱과 R²

#### 1. 상관계수 제곱과 R²

## 08 p-value 해석

### 1 p-value의 판단 기준

#### 1. p-value의 공통 판단 기준

#### 2. 분석별 p-value의 의미

### 2 변수·모형 차원의 해석

#### 1. 변수별 p-value와 모형 전체 p-value

변수별 p-value와 모형 전체 p-value를 구분해 출력해본다.

In [11]:
print('variable_p_values')
print(model_multi.pvalues)
print(
    'f_p_value: ',
    format(model_multi.f_pvalue, '.6e')
)

variable_p_values
const         1.337511e-19
ad_budget     4.953174e-43
is_weekend    5.571224e-05
dtype: float64
f_p_value:  4.672488e-42


#### 2. p-value와 효과 크기

#### 3. p-value 기준 변수 선택과 모형 재적합

문제에서 유의수준을 기준으로 변수를 선택한 뒤 모형을 다시 적합하라고 하면 절편을 제외한 변수별 p-value에서 조건에 맞는 컬럼명을 추출한다. p-value가 크다는 이유만으로 모든 분석에서 변수를 자동으로 제거하는 것은 아니며, 변수 선택을 요구한 경우에만 적용한다.

다음은 날짜에서 일자를 나타내는 day_of_month를 만든 뒤 광고비, 주말 여부와 함께 첫 번째 모형을 적합하는 예다.

In [12]:
df['date'] = pd.to_datetime(df['date'])
df['day_of_month'] = df['date'].dt.day
features = [
    'ad_budget',
    'is_weekend',
    'day_of_month'
]
X_full = sm.add_constant(
    df[features],
    has_constant='add'
)
y = df['orders']
full_model = sm.OLS(y, X_full).fit()
print(full_model.pvalues)

const           3.558639e-15
ad_budget       1.927193e-42
is_weekend      5.659762e-05
day_of_month    4.510580e-01
dtype: float64


유의수준 0.05에서 day_of_month의 p-value는 0.05보다 크다. 절편 const는 독립변수가 아니므로 변수 선택 대상에서 제외한다.

In [13]:
alpha = 0.05
feature_pvalues = full_model.pvalues.drop('const')
sig_vars = feature_pvalues[
    feature_pvalues < alpha
].index.tolist()
X_refit = sm.add_constant(
    df[sig_vars],
    has_constant='add'
)
refit_model = sm.OLS(y, X_refit).fit()
print('selected_variables: ', sig_vars)
print(refit_model.params.round(4))

selected_variables:  ['ad_budget', 'is_weekend']
const         83.0955
ad_budget      2.0163
is_weekend    18.9344
dtype: float64


## 09 신뢰구간 해석

### 1 회귀계수의 95% 신뢰구간

회귀계수의 95% 신뢰구간을 계수, p-value와 한 표로 정리해본다.

In [14]:
ci = model_multi.conf_int(alpha=0.05)
coef_table = pd.DataFrame({
    'coef': model_multi.params,
    'ci_lower': ci[0],
    'ci_upper': ci[1],
    'p_value': model_multi.pvalues
})
print(coef_table.round(4))

               coef  ci_lower  ci_upper  p_value
const       83.0955   69.4883   96.7027   0.0000
ad_budget    2.0163    1.8771    2.1554   0.0000
is_weekend  18.9344   10.0995   27.7692   0.0001


### 2 평균 예측 신뢰구간과 개별값 예측구간

같은 입력에 대해 평균 예측의 신뢰구간과 개별값 예측구간을 함께 확인해본다.

In [15]:
new_data = pd.DataFrame({
    'const': [1, 1],
    'ad_budget': [100, 100],
    'is_weekend': [0, 1]
})
pred = model_multi.get_prediction(new_data)
pred_table = pred.summary_frame(alpha=0.05)
print(pred_table[[
    'mean',
    'mean_ci_lower',
    'mean_ci_upper',
    'obs_ci_lower',
    'obs_ci_upper'
]].round(4))

       mean  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  284.7206       279.8644       289.5768      248.7757      320.6655
1  303.6549       295.9744       311.3355      267.2208      340.0891


## 10 회귀 결과표 해석

### 1 결과표 만들기와 읽기

#### 1. 결과표에서 확인하는 값

#### 2. 필요한 결과표 직접 만들기

필요한 값만 모아 결과표를 직접 만들어 본다.

In [16]:
summary_table = pd.DataFrame({
    'coef': model_multi.params,
    'std_err': model_multi.bse,
    't': model_multi.tvalues,
    'p_value': model_multi.pvalues,
    'ci_lower': model_multi.conf_int()[0],
    'ci_upper': model_multi.conf_int()[1]
})
print('nobs: ', int(model_multi.nobs))
print('df_model: ', int(model_multi.df_model))
print('df_resid: ', int(model_multi.df_resid))
print('r_squared: ', round(model_multi.rsquared, 4))
print(
    'adj_r_squared: ',
    round(model_multi.rsquared_adj, 4)
)
print('f_statistic: ', round(model_multi.fvalue, 4))
print('f_p_value: ', format(model_multi.f_pvalue, '.6e'))
print(summary_table.round(4))

nobs:  80
df_model:  2
df_resid:  77
r_squared:  0.9156
adj_r_squared:  0.9134
f_statistic:  417.5148
f_p_value:  4.672488e-42
               coef  std_err        t  p_value  ci_lower  ci_upper
const       83.0955   6.8335  12.1600   0.0000   69.4883   96.7027
ad_budget    2.0163   0.0699  28.8503   0.0000    1.8771    2.1554
is_weekend  18.9344   4.4368   4.2675   0.0001   10.0995   27.7692


#### 3. 결과표 읽는 순서

### 2 종합 해석과 답안 활용

#### 1. 이번 결과의 종합 해석

#### 2. 필요한 답안 값 추출

유의한 독립변수의 개수, 절편을 포함한 계수 합, 절편을 제외한 계수 합처럼 문제에서 요구한 값만 정확히 추출한다.

In [17]:
print('significant_count: ', len(sig_vars))
print(
    'coef_sum_with_const: ',
    round(refit_model.params.sum(), 4)
)
print(
    'coef_sum_without_const: ',
    round(refit_model.params.drop('const').sum(), 4)
)

significant_count:  2
coef_sum_with_const:  104.0461
coef_sum_without_const:  20.9506


#### 3. 선별된 변수로 새로운 값 예측

재적합한 모형으로 새로운 값을 예측하려면 새 데이터의 컬럼 구조를 재적합 모형의 X와 같게 맞춘다.

In [18]:
new_data = pd.DataFrame({
    'ad_budget': [100],
    'is_weekend': [1],
    'day_of_month': [15]
})
new_X = sm.add_constant(
    new_data,
    has_constant='add'
)
new_X = new_X.reindex(
    columns=X_refit.columns,
    fill_value=0
)
pred = refit_model.predict(new_X)
print('predicted_orders: ', round(pred.iloc[0], 4))

predicted_orders:  303.6549
